# Data Integration

**Objective:** Create a master dataset while keeping user origin and attraction location separate.

The code is split into visible, explainable steps for a demonstration video.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'Tourism Dataset'
PROCESSED = ROOT / 'data' / 'processed'
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

In [ ]:
names = ['transaction','user','city','country','region','continent','mode','type','item']
t = {name: pd.read_csv(PROCESSED/f'{name}_clean.csv') for name in names}
tx = t['transaction'].merge(t['mode'].rename(columns={'VisitMode':'VisitModeLabel'}), left_on='VisitMode', right_on='VisitModeId', how='left')
tx['VisitMode'] = tx['VisitModeLabel']
tx = tx.drop(columns=['VisitModeId','VisitModeLabel'])
tx.head()

In [ ]:
user_geo = t['user'].merge(t['city'].rename(columns={'CityId':'UserCityId','CityName':'UserCity'}), left_on='CityId', right_on='UserCityId', how='left')
user_geo = user_geo.merge(t['country'].rename(columns={'Country':'UserCountry','RegionId':'UserRegionId'}), on='CountryId', how='left').merge(t['region'].rename(columns={'RegionId':'UserRegionId','Region':'UserRegion','ContinentId':'UserContinentId'}), on='UserRegionId', how='left').merge(t['continent'].rename(columns={'ContinentId':'UserContinentId','Continent':'UserContinent'}), on='UserContinentId', how='left')
user_geo = user_geo[['UserId','UserCity','UserCountry','UserRegion','UserContinent']]
user_geo.head()

In [ ]:
attraction_geo = t['item'].merge(t['type'], on='AttractionTypeId', how='left').merge(t['city'].rename(columns={'CityId':'AttractionCityId','CityName':'AttractionCity','CountryId':'AttractionCountryId'}), on='AttractionCityId', how='left')
attraction_geo = attraction_geo.merge(t['country'], left_on='AttractionCountryId', right_on='CountryId', how='left').merge(t['region'], on='RegionId', how='left').merge(t['continent'], on='ContinentId', how='left').rename(columns={'Country':'AttractionCountry','Region':'AttractionRegion','Continent':'AttractionContinent'})
attraction_geo = attraction_geo[['AttractionId','Attraction','AttractionType','AttractionCity','AttractionCountry','AttractionRegion','AttractionContinent','AttractionAddress']]

In [ ]:
master = tx.merge(user_geo, on='UserId', how='left', validate='many_to_one').merge(attraction_geo, on='AttractionId', how='left', validate='many_to_one')
assert len(master) == len(tx), 'Unexpected row multiplication'
master.to_csv(PROCESSED/'master_dataset.csv', index=False)
master.head()